In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv('ODI data.csv')

# Remove unnecessary columns
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 13'])

# Clean HS column (remove * symbol)
df['HS'] = df['HS'].astype(str).str.replace('*', '', regex=False)

# Convert columns to numeric
num_cols = ['Mat', 'Inns', 'NO', 'Runs', 'HS', 'Ave', 'BF', 'SR', '100', '50', '0']

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove rows with missing values
df = df.dropna()

# Separate features and target
X = df.drop('Runs', axis=1)
y = df['Runs']

print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

Shape of X: (2369, 12)
Shape of y: (2369,)


In [3]:
# Identify categorical columns
cat_cols = X.select_dtypes(include='object').columns.tolist()
print(cat_cols)

# One-Hot Encoding
X_encoded = pd.get_dummies(X, columns=cat_cols).astype(int)

# Show first 5 rows
print(X_encoded.head())

['Player', 'Span']
   Mat  Inns  NO   HS  Ave     BF  SR  100  50   0  ...  Span_2016-2016  \
0  463   452  41  200   44  21367  86   49  96  20  ...               0   
1  404   380  41  169   41  18048  78   25  93  15  ...               0   
2  375   365  39  164   42  17046  80   30  82  20  ...               0   
3  445   433  18  189   32  14725  91   28  68  34  ...               0   
4  448   418  39  144   33  16020  78   19  77  28  ...               0   

   Span_2016-2017  Span_2016-2018  Span_2016-2019  Span_2017-2017  \
0               0               0               0               0   
1               0               0               0               0   
2               0               0               0               0   
3               0               0               0               0   
4               0               0               0               0   

   Span_2017-2018  Span_2017-2019  Span_2018-2018  Span_2018-2019  \
0               0               0             

C:\Users\chede appliances\AppData\Local\Temp\ipykernel_13244\3454867524.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns.tolist()


In [4]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['Mat', 'Inns', 'NO', 'HS', 'Ave', 'BF', 'SR', '100', '50', '0']

scaler = StandardScaler()

X_scaled = X_encoded.copy()
X_scaled[scale_cols] = scaler.fit_transform(X_scaled[scale_cols])

print(X_scaled[scale_cols].head())

        Mat      Inns        NO        HS       Ave        BF        SR  \
0  7.076497  8.054977  3.696644  3.402219  2.044457  9.704142  0.901867   
1  6.091670  6.677266  3.696644  2.699331  1.809234  8.126578  0.586501   
2  5.607603  6.390243  3.487188  2.585962  1.887641  7.650314  0.665343   
3  6.776041  7.691414  1.287901  3.152807  1.103565  6.547111  1.098970   
4  6.826117  7.404391  3.487188  2.132486  1.181973  7.162642  0.586501   

         100        50         0  
0  16.043121  9.146432  4.463955  
1   8.061011  8.848981  3.181191  
2   9.723950  7.758329  4.463955  
3   9.058774  6.370225  8.055695  
4   6.065483  7.262577  6.516378  


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.33,
    random_state=42
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

X_train shape: (1587, 2943)
X_test shape: (782, 2943)
y_train shape: (1587,)
y_test shape: (782,)


In [6]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

print('Intercept:', model.intercept_)
print('First 10 coefficients:')
print(model.coef_[:10])

Intercept: 707.9455430759083
First 10 coefficients:
[-204.54500568  905.71277782   -3.99037693   16.47548345    3.32674943
  191.73102742    8.79576174  309.23964704  575.91820623  -91.360031  ]


In [7]:
y_pred = model.predict(X_test)

print('Predicted values:')
print(y_pred[:10])

print('\nActual values:')
print(y_test.iloc[:10].values)

Predicted values:
[ 1.77612686e+02  2.37862870e+03  1.68160510e+01  4.87639215e+02
  3.41750712e+02  4.23122529e-02  2.33995203e+03  3.20781327e+01
  1.11181782e+03 -2.98523516e+01]

Actual values:
[ 141. 2434.   26.  368.  401.   16. 2204.   24. 1283.   24.]


In [8]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)

print('R² Score:', r2)

R² Score: 0.9961430096704708


In [ ]:
import joblib

# Save model
joblib.dump(model, 'LR_ODI.pkl')

# Save scaler
joblib.dump(scaler, 'scaler.pkl')

# Save columns
joblib.dump(X_scaled.columns.tolist(), 'columns.pkl')

print('Model and preprocessing files saved successfully.')

Model and preprocessing files saved successfully.
